In [66]:
# PBL6 김재영
import os
import re
import time

class Monitoring:

    def __init__(self, monitoring_file_path):
  
       # 파일 시스템 모니터링 이해, 초기화
        self.monitoring_file_path = monitoring_file_path
        
        # 디렉토리가 없을 경우 예외 처리
        if not os.path.exists(self.monitoring_file_path): # 경로 파일 확인 
            os.makedirs(self.monitoring_file_path) # 디렉토리 생성
        
        self.before_f = set(os.listdir(self.monitoring_file_path)) #경로안의 파일들을 리스트 형태로 가져온 뒤 set형식으로 변환

        # 특정 확장자 주의 파일 분류
        self.warning_extensions = {'.js', '.class', '.php', '.py', '.exe', '.ipynb', '.sh'}
        
        # 주요 정보 검증패턴 정의 (정규표현식)
        self.search_patterns = {
            '주석': r'#.*|//.*',                                              #주석 형식 검증
            'Email': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',       #이메일 주소 형식 검증 
            'SQL 쿼리': r'\b(INSERT|SELECT|UPDATE|DELETE|CREATE|DROP)\b',        # SQL 주요 쿼리문 검증 
            '보안 관련 키워드': r'\b(key|token|secret|admin|password|id)\b'          #보안관련 키워드 검증
        }

    def analyze(self, f_name):
        # 파일 내용 분석 및 줄 번호 출력
        f_path = os.path.join(self.monitoring_file_path, f_name)
        print(f' 파일 내용 분석| 파일명: {f_name}')
        try:

            with open(f_path, 'r', encoding='utf-8', errors='ignore') as f:
                for num, line in enumerate(f, 1):
                    line = line.strip()
                    if not line:
                        continue

                    # 패턴을 하나씩 검사
                    for label, pattern in self.search_patterns.items():
                        # re.search의 결과를 match 변수에 담기
                        match = re.search(pattern, line, re.IGNORECASE) #대소문자 구분X 패턴 매칭
                                
                        if match:
                            found_text = match.group()    
                            if label == '주석 처리':
                                # #뒤의 내용을 무시하고 '#'만 출력
                                print(f' -> {label} 발견 (Line {num}): #')
                            elif label == 'SQL 쿼리문':
                                # 매칭된 SQL 단어(예: DELETE)만 출력
                                print(f' -> {label} 발견 (Line {num}): {found_text}')
                            else:
                                # 나머지는 기존처럼 매칭된 부분 출력
                                print(f' -> {label} 발견 (Line {num}): {found_text}')
                            
        except Exception as e: # 에러발생 예외처리
            print(f'에러 발생 {e}')

    def repetition(self):
        # 모니터링 시작 안내
        print("모니터링 파일 위치: ", self.monitoring_file_path)
        print(f"실행시점 파일의 개수: {len(self.before_f)}개" )
        try:
            # 계속 반복하면서 폴더를 확인
            while 1:
                # 현재 폴더에 있는 파일 목록 가져오기
                now_f = set(os.listdir(self.monitoring_file_path)) # 현재 파일

                # 이전 파일 목록과 비교해서 새로 생긴 파일 찾기
                new_f = now_f - self.before_f

                #  파일 목록과 비교해서 사라진 파일 찾기
                delete_f = self.before_f - now_f

                # 새 파일이 생성된 경우
                if new_f:
                    for f_name in new_f:
                        # 전체 경로
                        full_path = os.path.join(self.monitoring_file_path, f_name)

                        # 파일인지 확인 (폴더 제외)
                        if os.path.isfile(full_path):

                            print("\n새 파일 확인: ", f_name)

                            # 파일 확장자 확인
                            ext = os.path.splitext(f_name)[1].lower()# 확장자가 대문자일경우 정상 처리되게끔 lower()사용
                            
                            #관리 대상 확장자일때
                            if ext in self.warning_extensions:
                                print('---!!주의 파일 분류: ', ext) # 관리대상 확장자명 출력
                                    
                            self.analyze(f_name)

                        else:
                            print("\n새 폴더 생성:", f_name, "(분석 제외)") # 새 폴더 생성은 분석제외
                
                # 증감, 유지 확인
                if new_f or delete_f:
                    before_count = len(self.before_f)
                    now_count = len(now_f)

                    status = '증가' if now_count > before_count else '감소' if now_count < before_count else '유지(이름변경)'
                    print(f'\n파일 개수 변동({status}): {before_count}개 -> {now_count}개')
                    
                    # 기준 파일 목록을 현재 목록으로 업데이트
                    self.before_f = now_f
                
                # 0.5초 간격으로 확인
                time.sleep(0.5)

        except KeyboardInterrupt: # 셀 중지시 예외처리
            print("\n모니터링 종료")

def main():
    # 과제에서 제시한 경로 설정
    target_dir = r'C:\SK_Shieldus_Rookies' #C:\SK_Shieldus_Rookies <- 내 파일 경로

    #target_dir 코드에 경로를 입력하지 않으면 실행
    while 1:
        if not target_dir.strip():
            print('\n경로가 입력되지 않았습니다.')
            target_dir = input('경로를 입력해주세요(q 입력시 종료): ').strip()

        if target_dir.lower() == 'q':
            print('프로그램 종료')
            return
                  
        try:
            if not os.path.exists(target_dir):
                os.makedirs(target_dir)
                print(f'새 디렉토리 생성: {target_dir}')
            else:
                print(f'기존 디렉토리 확인: {target_dir}')
            break

        except Exception as e : 
            print(f'경로가 유효하지 않습니다.: {e}')
            target_dir = '' # 다시 입력 받기위해 초기화
            continue
    
    # 경로가 유효할때 객체 생성 및 실행
    if target_dir and target_dir.lower() != 'q':
        scanner = Monitoring(target_dir)
        scanner.repetition()

#실행
main()

기존 디렉토리 확인: C:\SK_Shieldus_Rookies
모니터링 파일 위치:  C:\SK_Shieldus_Rookies
실행시점 파일의 개수: 37개

새 파일 확인:  12.txt
 파일 내용 분석| 파일명: 12.txt
 -> 주석 발견 (Line 1): #
 -> 주석 발견 (Line 2): ##
 -> 주석 발견 (Line 3): ####
 -> SQL 쿼리 발견 (Line 4): delete
 -> 주석 발견 (Line 5): ####

파일 개수 변동(유지(이름변경)): 37개 -> 37개

새 파일 확인:  검사용입니다.txt
 파일 내용 분석| 파일명: 검사용입니다.txt
 -> 주석 발견 (Line 1): #
 -> 주석 발견 (Line 2): ##
 -> 주석 발견 (Line 3): ####
 -> SQL 쿼리 발견 (Line 4): delete
 -> 주석 발견 (Line 5): ####
 -> 보안 관련 키워드 발견 (Line 6): key
 -> 보안 관련 키워드 발견 (Line 8): token

파일 개수 변동(유지(이름변경)): 37개 -> 37개

모니터링 종료
